# Featherweight — Phase 4 Group C: fine-tuned (base + LoRA) BFCL eval (Colab T4)

Runs the **fine-tuned** model through the *same* BFCL harness + `SalesforceLlamaHandler`
prompting contract as the base baseline, so the base-vs-FT delta is apples-to-apples.
The adapter is served **on top of** the bnb-4bit base via vLLM `--enable-lora` — no
merge/quantize needed (that's the Phase 6 serving track).

**Contract:**
- Register `featherweight-ft` with `model_name="featherweight-ft"` so bfcl's request
  `model` field matches the vLLM `--lora-modules` name and routes to the **adapter**.
- The tokenizer can't load from that non-HF id, so we set
  `REMOTE_OPENAI_TOKENIZER_PATH` to the real base id (and `REMOTE_OPENAI_BASE_URL` —
  both are required together for bfcl to honor the tokenizer override).

> **Runtime risk:** LoRA on a bnb-4bit base on a T4 (Turing) is the one unverified
> path. If vLLM rejects it at load time, see the fallback note in the serve cell.

## 1. GPU + install

In [ ]:
!nvidia-smi

In [ ]:
# vLLM (server) + bfcl-eval (harness) + soundfile (dep bfcl doesn't pin).
# Pin transformers<5: bfcl's deps pull transformers 5.x which breaks vLLM's
# tokenizer (TokenizersBackend has no all_special_tokens_extended).
!pip install -q vllm 'bfcl-eval==2026.3.23' soundfile 'transformers<5' 'bitsandbytes>=0.46.1'  # bitsandbytes: vLLM bnb quantizer

## 2. Repo + package

In [ ]:
import os

REPO = "/content/Featherweight"
![ -d {REPO}/.git ] || git clone https://github.com/Nishaant-Soni/Featherweight {REPO}
%cd {REPO}
!pip install -q -e .
import sys

sys.path.insert(0, f"{REPO}/src")  # editable .pth is read only at startup
from featherweight import config

print("ROOT_DIR:", config.ROOT_DIR)
CATS = ",".join(config.CONFIG.eval.categories)
print("categories:", CATS)

## 3. BFCL project root + serving env
`BFCL_PROJECT_ROOT` is where bfcl writes `result/`+`score/` (kept under the repo).
`REMOTE_OPENAI_*` decouple the **tokenizer source** (real base id) from the request
`model` id (the LoRA name) — both vars are required together (see iteration_13).

In [ ]:
import os

BASE_ID = config.BASE_MODEL_4BIT  # unsloth/llama-3.1-8b-Instruct-bnb-4bit (ungated)
os.environ["BFCL_PROJECT_ROOT"] = f"{REPO}/third_party/bfcl"
os.environ["LOCAL_SERVER_ENDPOINT"] = "localhost"
os.environ["LOCAL_SERVER_PORT"] = "8000"
# Route requests to the adapter (model=featherweight-ft) but load the tokenizer/config
# from the real base id. bfcl only honors the tokenizer override when BOTH are set.
os.environ["REMOTE_OPENAI_BASE_URL"] = "http://localhost:8000/v1"
os.environ["REMOTE_OPENAI_TOKENIZER_PATH"] = BASE_ID
os.makedirs(os.environ["BFCL_PROJECT_ROOT"], exist_ok=True)
print("BFCL_PROJECT_ROOT =", os.environ["BFCL_PROJECT_ROOT"])
print("tokenizer path    =", BASE_ID)

## 4. Download the LoRA adapter (private HF repo)
`snapshot_download` returns the local dir (with `adapter_config.json` +
`adapter_model.safetensors`) that vLLM `--lora-modules` points at. Private repo →
paste an HF **read** token when prompted.

In [ ]:
from huggingface_hub import login, snapshot_download

login(token="your_huggingface_token_here")  # replace with your actual token
ADAPTER_DIR = snapshot_download("your_username/featherweight-adapter")  # repo_type=model
print("adapter at:", ADAPTER_DIR)

## 5. Launch vLLM with the LoRA adapter (background)
Serves the bnb-4bit base + the adapter under the name `featherweight-ft`. `--dtype half`
(T4 has no bf16); `--max-lora-rank` = the trained LoRA r.

**Fallback if LoRA-on-bnb fails to load:** (a) merge the adapter into the base then
quantize (Phase 6's `merge_quantize`) and serve that as a plain model (drop
`--enable-lora`), or (b) serve fp16 base + LoRA with a lower `--max-model-len` to fit 16 GB.

In [ ]:
!pkill -f "vllm serve" 2>/dev/null; sleep 3

import subprocess, time, requests

LORA_RANK = config.CONFIG.train.lora.r  # 16 — single source of truth
# log to a file so we can see progress/errors (subprocess stdout isn't shown in the cell)
logf = open("/content/vllm.log", "w")
server = subprocess.Popen(
    [
        "vllm",
        "serve",
        BASE_ID,
        "--quantization",
        "bitsandbytes",
        "--load-format",
        "bitsandbytes",
        "--dtype",
        "half",
        "--max-model-len",
        "8192",
        "--gpu-memory-utilization",
        "0.9",
        "--port",
        "8000",
        "--enable-lora",
        "--lora-modules",
        f"featherweight-ft={ADAPTER_DIR}",
        "--max-lora-rank",
        str(LORA_RANK),
    ],
    stdout=logf,
    stderr=subprocess.STDOUT,
)
print("launched pid", server.pid)
for i in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).status_code == 200:
            print("vLLM server ready")
            break
    except Exception:
        pass
    print(f"--- waiting ({i * 5}s); last log lines: ---")
    !tail -n 4 /content/vllm.log
    time.sleep(5)
else:
    !tail -n 40 /content/vllm.log
    raise RuntimeError("vLLM server did not become ready - see log above")

## 6. Generate (FT, via our own server)
Same fast driver as the base run: registers `featherweight-ft` and **clamps each
completion to 512 tokens** (tool-call arrays are short). `--num-threads 32` so vLLM
batches. No `--allow-overwrite` — a fresh run; to resume a partial run later, re-run
this cell *without* `--allow-overwrite` and bfcl skips already-generated cases.

In [ ]:
%%writefile /content/run_bfcl_ft_fast.py
# Register featherweight-ft, then run the bfcl CLI with completions clamped to 512 tokens.
from featherweight.eval import bfcl_register
bfcl_register.register_ft_model()

from openai.resources.completions import Completions
_orig = Completions.create
def _capped(self, *args, **kwargs):
    mt = kwargs.get('max_tokens')
    if mt is None or mt > 512:
        kwargs['max_tokens'] = 512
    return _orig(self, *args, **kwargs)
Completions.create = _capped

from bfcl_eval.__main__ import cli
cli()

In [7]:
!python /content/run_bfcl_ft_fast.py generate --model featherweight-ft \
    --backend vllm --skip-server-setup --num-threads 32 --test-category {CATS}

Generating results for ['featherweight-ft']
Running full test cases for categories: ['irrelevance', 'multiple', 'parallel', 'parallel_multiple', 'simple_python'].
Loaded tokenizer from REMOTE_OPENAI_TOKENIZER_PATH: unsloth/llama-3.1-8b-Instruct-bnb-4bit
Max context length: 131072
server is ready!
Generating results for featherweight-ft: 100% 1240/1240 [1:55:34<00:00,  5.59s/it] 


## 7. Evaluate (CPU scoring)

In [8]:
!python scripts/run_bfcl.py evaluate --model featherweight-ft --test-category {CATS}

Number of models evaluated:   0% 0/1 [00:00<?, ?it/s]🦍 Model: featherweight-ft
🔍 Running test: multiple
✅ Test completed: multiple. 🎯 Accuracy: 93.50%
🔍 Running test: parallel
✅ Test completed: parallel. 🎯 Accuracy: 92.00%
🔍 Running test: irrelevance
✅ Test completed: irrelevance. 🎯 Accuracy: 76.25%
🔍 Running test: simple_python
✅ Test completed: simple_python. 🎯 Accuracy: 93.75%
🔍 Running test: parallel_multiple
✅ Test completed: parallel_multiple. 🎯 Accuracy: 90.00%
Number of models evaluated: 100% 1/1 [00:00<00:00,  1.82it/s]
📈 Aggregating data to generate leaderboard score table...
🏁 Evaluation completed. See /content/Featherweight/third_party/bfcl/score/data_overall.csv for overall evaluation results on BFCL V4.
See /content/Featherweight/third_party/bfcl/score/data_live.csv, /content/Featherweight/third_party/bfcl/score/data_non_live.csv, /content/Featherweight/third_party/bfcl/score/data_multi_turn.csv, /content/Featherweight/third_party/bfcl/score/data_agentic.csv and /content/

## 8. Parse scores + stop the server
Paste the printed per-category numbers back to combine with base + GPT-4o via
`report.write_baselines` into the three-way `results/baselines.{csv,md}`.

In [9]:
from pathlib import Path
from featherweight.eval import report

score_dir = Path(os.environ["BFCL_PROJECT_ROOT"]) / "score" / "featherweight-ft"
for cat, s in report.collect_scores(score_dir).items():
    print(cat, s)

# zip the score dir for safekeeping (HF upload is more reliable than files.download here)
import shutil

shutil.make_archive("/content/ft_scores", "zip", score_dir)
print("scores zipped -> /content/ft_scores.zip")
server.terminate()

irrelevance {'accuracy': 0.7625, 'correct_count': 183, 'total_count': 240, 'invalid_rate': 0.0}
multiple {'accuracy': 0.935, 'correct_count': 187, 'total_count': 200, 'invalid_rate': 0.005}
parallel_multiple {'accuracy': 0.9, 'correct_count': 180, 'total_count': 200, 'invalid_rate': 0.005}
parallel {'accuracy': 0.92, 'correct_count': 184, 'total_count': 200, 'invalid_rate': 0.005}
simple_python {'accuracy': 0.9375, 'correct_count': 375, 'total_count': 400, 'invalid_rate': 0.005}
scores zipped -> /content/ft_scores.zip
